# Cross-dataset test (CREMA-D) and the CPU model

1. **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all** (about 40 minutes).
2. Downloads RAVDESS (200 MB, CC BY-NC-SA 4.0) and CREMA-D's 7,442 audio clips (about 550 MB,
   ODbL) into this Colab machine only.
3. `scripts/cross_dataset.py`: train on one corpus, test on the other, for their six shared emotions.
4. `scripts/export_onnx.py`: WavLM to ONNX, int8 quantisation, the served head, accuracy and latency checks.

In [ ]:
BRANCH = "feat/cross-dataset-onnx"
%cd /content
!rm -rf voice-emotion-detection
!git clone -q -b $BRANCH https://github.com/CODER7657/voice-emotion-detection.git
%cd /content/voice-emotion-detection
!pip -q install -e ".[ssl,export,serve]" onnxscript 2>&1 | tail -2
import sys
sys.path.insert(0, "/content/voice-emotion-detection/src")

In [ ]:
import os, json, urllib.request
from concurrent.futures import ThreadPoolExecutor
if not os.path.isdir("/content/ravdess/Actor_24"):
    !wget -q -O /content/ravdess.zip "https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1"
    !unzip -q -o /content/ravdess.zip -d /content/ravdess
# CREMA-D audio, file by file from the dataset's GitHub repository (Git LFS media).
os.makedirs("/content/cremad", exist_ok=True)
tree = json.load(urllib.request.urlopen("https://api.github.com/repos/CheyneyComputerScience/CREMA-D/git/trees/master?recursive=1"))
names = [t["path"].split("/")[-1] for t in tree["tree"] if t["path"].startswith("AudioWAV/") and t["path"].endswith(".wav")]
def fetch(name):
    dst = f"/content/cremad/{name}"
    if not os.path.exists(dst) or os.path.getsize(dst) < 1000:
        for _ in range(4):
            try:
                urllib.request.urlretrieve(f"https://media.githubusercontent.com/media/CheyneyComputerScience/CREMA-D/master/AudioWAV/{name}", dst)
                break
            except Exception:
                pass
    return os.path.exists(dst) and os.path.getsize(dst) > 1000
with ThreadPoolExecutor(32) as ex:
    ok = sum(ex.map(fetch, names))
urllib.request.urlretrieve("https://raw.githubusercontent.com/CheyneyComputerScience/CREMA-D/master/VideoDemographics.csv", "/content/VideoDemographics.csv")
print(len(names), "listed,", ok, "downloaded")

In [ ]:
!PYTHONPATH=src python scripts/cross_dataset.py --ravdess /content/ravdess --cremad /content/cremad --demographics /content/VideoDemographics.csv --out results/cross_dataset 2>&1 | grep -v Warning

In [ ]:
!PYTHONPATH=src python scripts/export_onnx.py --ravdess /content/ravdess --out models 2>&1 | grep -v Warning
!ls -la models